# SVM Lineal — Riesgo cardiovascular (BRFSS 2024)

Este cuaderno implementa el modelo SVM Lineal. Se emplean técnicas de validación cruzada, imputación proporcional, normalización y SMOTE.




## Importar bibliotecas

El siguiente script importa las bibliotecas necesarias para trabajar con el modelo.

In [3]:
import os
import sys
import time
sys.path.append("..")

import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from scripts import pipeline as pl

Los datos de entrada ya vienen imputados, normalizados y balanceados con SMOTENC, generados una sola vez por scripts/preprocesar_pliegues.py (ver README del proyecto). Si la carpeta data_procesada/ no existe, es necesario correr ese script antes de continuar.

In [25]:
DATA_DIR = "../data_procesada"
RESULTS_DIR = "../results"
N_PLIEGUES = 5


def cargar(nombre):
    """Lee una particion ya preprocesada (generada por scripts/preprocesar_pliegues.py)."""
    tabla = pd.read_parquet(f"{DATA_DIR}/{nombre}")
    X = tabla.drop(columns=[pl.TARGET_COL])
    y = tabla[pl.TARGET_COL]
    return X, y

In [7]:
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        "No se encontro data_procesada/. Corre primero 'python scripts/preprocesar_pliegues.py' "
        "desde la raiz del proyecto (tarda unos 15 minutos, se corre una sola vez por maquina)."
    )

print("cache encontrado en", DATA_DIR)

cache encontrado en ../data_procesada


## Prueba sobre el pliegue 0
Antes de ejecutar el ciclo completo de 5 pliegues, se valida el procedimiento sobre un único pliegue, con el fin de confirmar que el modelo entrena correctamente sobre estos datos y que la función pipeline.evaluar() procesa sus predicciones sin errores.

In [10]:
X_train, y_train = cargar("fold_0_train.parquet")
X_test, y_test = cargar("fold_0_test.parquet")
prot_test = X_test[pl.PROTECTED_COLS]

print("train:", X_train.shape, "| balance:", y_train.value_counts().to_dict())

train: (634536, 30) | balance: {0: 317268, 1: 317268}


In [11]:
modelo = CalibratedClassifierCV(estimator=LinearSVC(C=1.0, random_state=pl.RANDOM_STATE, dual=False,),
    ensemble=False)

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

In [12]:
resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)

print("RESUMEN (pliegue 0):")
for k, v in resumen.items():
    print(f"  {k}: {v:.4f}")

detalle["_RACE"]

RESUMEN (pliegue 0):
  auroc: 0.7882
  eod_sexo: 0.0903
  fpr_disp_sexo: 0.0409
  calib_desv_sexo: 2.3079
  eod_raza: 0.5612
  fpr_disp_raza: 0.2604
  calib_desv_raza: 3.1295
  eod_ingreso: 0.2771
  fpr_disp_ingreso: 0.2488
  calib_desv_ingreso: 3.0088


,grupo,n,tpr,fpr,calibracion
0,1.0,66555,0.730519,0.292510,3.070906
1,2.0,6987,0.590613,0.204180,2.922636
2,4.0,2469,0.380531,0.056027,4.129540
3,7.0,2053,0.169355,0.032133,1.307400
4,8.0,9544,0.381032,0.076820,3.243482
5,6.0,701,0.300000,0.057283,1.716560
6,3.0,1248,0.491228,0.142061,2.211814
7,5.0,395,0.319149,0.066092,1.715392


## Ajuste de hiperparámetros

El parámetro más importante para entrenar al modelo es C. Controla el equilibrio entre el margen de separación entre clases y los errores de clasificación en los datos de entrenamiento. Aumenta o disminuye en potencias de 10 (0.001, 0.01, 0.1, 1...)

Valores pequeños de C permiten una mayor tolerancia a errores de clasificación, mientras que un valor alto de C penaliza mucho más los errores de clasificación. Se debe ajustar el hiperparámetro para maximizar el AUROC.


In [14]:
X_tr_int, y_tr_int = cargar("tuning_train_interno.parquet")
X_val_int, y_val_int = cargar("tuning_validacion_interna.parquet")

print("train interno:", X_tr_int.shape, "| validacion interna:", X_val_int.shape)

train interno: (507628, 30) | validacion interna: (71962, 30)


In [35]:
grid = [dict(C=C)
    for C in [0.001, 0.01, 0.1, 1, 10, 100]
]

resultados_grid = []


for params in grid:

    inicio = time.time()
    modelo_grid = CalibratedClassifierCV(LinearSVC(**params,dual=False,random_state=pl.RANDOM_STATE,), ensemble = False)
    modelo_grid.fit(X_tr_int, y_tr_int)
    proba_val = modelo_grid.predict_proba(X_val_int)[:, 1]
    duracion = time.time() - inicio
    resultados_grid.append({**params,"auroc_validacion": roc_auc_score(y_val_int, proba_val),"segundos": round(duracion)})
    print(params, f"AUROC={resultados_grid[-1]['auroc_validacion']:.4f} ({duracion:.0f}s)")


tabla_grid = pd.DataFrame(resultados_grid).sort_values("auroc_validacion", ascending=False)
tabla_grid.to_csv(f"{RESULTS_DIR}/grid_svmlineal.csv", index=False)
tabla_grid.round(4)

{'C': 0.001} AUROC=0.7896 (84s)
{'C': 0.01} AUROC=0.7887 (154s)
{'C': 0.1} AUROC=0.7884 (193s)
{'C': 1} AUROC=0.7884 (189s)
{'C': 10} AUROC=0.7883 (200s)
{'C': 100} AUROC=0.7883 (197s)


,C,auroc_validacion,segundos
0,0.001,0.7896,84
1,0.010,0.7887,154
2,0.100,0.7884,193
3,1.000,0.7884,189
5,100.000,0.7883,197
4,10.000,0.7883,200


In [36]:
mejor = tabla_grid.iloc[0]
MEJORES_HIPERPARAMETROS = dict(
    C=float(mejor["C"]),
)
MEJORES_HIPERPARAMETROS

{'C': 0.001}

## Ciclo completo de validación cruzada (5 pliegues)

Una vez confirmado el funcionamiento sobre el pliegue 0, se repite el mismo procedimiento de forma independiente para cada uno de los 5 pliegues, y se reporta el promedio y la desviación estándar de cada métrica entre pliegues

In [30]:
resultados = []
detalle_total = []

for i in range(N_PLIEGUES):
    X_train, y_train = cargar(f"fold_{i}_train.parquet")
    X_test, y_test = cargar(f"fold_{i}_test.parquet")
    prot_test = X_test[pl.PROTECTED_COLS]

    modelo = CalibratedClassifierCV(LinearSVC(**MEJORES_HIPERPARAMETROS,dual=False,random_state=pl.RANDOM_STATE,), ensemble = False)
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)
    resumen["modelo"] = "svmlineal"
    resumen["pliegue"] = i
    resultados.append(resumen)

    for col, tabla_grupo in detalle.items():
        tabla_grupo = tabla_grupo.copy()
        tabla_grupo["atributo"] = pl.SUFIJOS_PROTEGIDAS[col]
        tabla_grupo["modelo"] = "svmlineal"
        tabla_grupo["pliegue"] = i
        detalle_total.append(tabla_grupo)

    print(f"pliegue {i} listo | AUROC = {resumen['auroc']:.4f}")

pliegue 0 listo | AUROC = 0.7890
pliegue 1 listo | AUROC = 0.7905
pliegue 2 listo | AUROC = 0.7908
pliegue 3 listo | AUROC = 0.7876
pliegue 4 listo | AUROC = 0.7861


In [37]:
tabla_metricas = pd.DataFrame(resultados)
columnas_ordenadas = ["modelo", "pliegue"] + [c for c in tabla_metricas.columns if c not in ("modelo", "pliegue")]
tabla_metricas = tabla_metricas[columnas_ordenadas]

resumen_cv = tabla_metricas.drop(columns=["modelo", "pliegue"]).agg(["mean", "std"]).round(4)

tabla_metricas.round(4)

,modelo,pliegue,auroc,eod_sexo,fpr_disp_sexo,calib_desv_sexo,eod_raza,fpr_disp_raza,calib_desv_raza,eod_ingreso,fpr_disp_ingreso,calib_desv_ingreso
0,svmlineal,0,0.7890,0.0908,0.0409,2.3105,0.5363,0.2553,3.1941,0.2908,0.2479,3.0069
1,svmlineal,1,0.7905,0.1161,0.0459,2.2228,0.5487,0.2608,2.9332,0.2542,0.2586,2.9551
2,svmlineal,2,0.7908,0.0979,0.0401,2.2588,0.5690,0.2588,2.8892,0.2723,0.2567,3.2480
3,svmlineal,3,0.7876,0.1122,0.0404,2.3002,0.5113,0.2589,4.0114,0.2538,0.2681,2.8657
4,svmlineal,4,0.7861,0.1062,0.0378,2.2907,0.5696,0.2704,2.9167,0.3003,0.2640,2.8652


In [32]:
resumen_cv

,auroc,eod_sexo,fpr_disp_sexo,calib_desv_sexo,eod_raza,fpr_disp_raza,calib_desv_raza,eod_ingreso,fpr_disp_ingreso,calib_desv_ingreso
mean,0.7888,0.1047,0.041,2.2766,0.5470,0.2608,3.1889,0.2743,0.2591,2.9882
std,0.0020,0.0103,0.003,0.0358,0.0244,0.0057,0.4759,0.0211,0.0077,0.1574


In [38]:
import os

os.makedirs("../results", exist_ok=True)
tabla_metricas.to_csv("../results/metrics_svmlineal.csv", index=False)
print("Guardado en results/metrics_svmlineal.csv")

Guardado en results/metrics_svmlineal.csv
